# Extra Verification

We are claiming that $(2,3,4,5,4,6,4,1)$ is minimal and filling, so we check that this is truly the case with a more manual evaluation and larger choice of prime than inour search. We use $p = 2^{13} - 1=8191$.

In [ ]:
from sage.all import *
import itertools
import numpy

class Network(object):

    def __init__(self, sizes, exponent=2, finite_field=100003):
        """The list ``sizes`` contains the number of neurons in the
        respective layers of the network.  For example, if the list
        was [2, 3, 1] then it would be a three-layer network, with the
        first layer containing 2 neurons, the second layer 3 neurons,
        and the third layer 1 neuron.  Currently, biases are initialized to 
        zero and weights are initialized randomly."""
        self.num_layers = len(sizes)
        self.sizes = sizes
        self.FF = GF(finite_field)
        self.biases = [zero_matrix(self.FF, y, 1) for y in sizes[1:]]  # [random_matrix(FF, y, 1) for y in sizes[1:]]
        self.weights = [random_matrix(self.FF, y, x) for x, y in zip(sizes[:-1], sizes[1:])]
        self.exponent = exponent
        self.degree = self.exponent**(self.num_layers - 2)

    def feedforward(self, a):
        """Return the output of the network if ``a`` is input."""
        for b, w in zip(self.biases[:-1], self.weights[:-1]):
            a = matrix_power(w * a + b, self.exponent)
        return self.weights[-1] * a + self.biases[-1]

    def backprop(self, x, pullback_vector=None):
        """Return a tuple ``(nabla_b, nabla_w)`` representing the
        gradient for the *output*pullback_vector* function.  ``nabla_b`` and
        ``nabla_w`` are layer-by-layer lists of matrices, similar
        to ``self.biases`` and ``self.weights``."""

        if pullback_vector is None:
            pullback_vector = ones_matrix(self.FF, self.sizes[-1], 1)

        nabla_b = [zero_matrix(self.FF, b.nrows(), b.ncols()) for b in self.biases]
        nabla_w = [zero_matrix(self.FF, w.nrows(), w.ncols()) for w in self.weights]
        # feedforward
        activation = x
        activations = [x]  # list to store all the activations, layer by layer
        zs = []  # list to store all the z vectors, layer by layer
        for b, w in zip(self.biases, self.weights):
            z = w * activation + b
            zs.append(z)
            activation = matrix_power(z, self.exponent)
            activations.append(activation)
        # backward pass
        delta = pullback_vector
        nabla_b[-1] = delta
        nabla_w[-1] = delta * activations[-2].T
        for l in range(2, self.num_layers):
            z = zs[-l]
            sp = matrix_power_prime(z, self.exponent)
            delta = elementwise_product(self.weights[-l + 1].T * delta, sp)
            nabla_b[-l] = delta
            nabla_w[-l] = delta * activations[-l - 1].T
        return (nabla_b, nabla_w)


def matrix_power(M, exponent=2):
    """Raise elements in M to the power exponent."""
    nc, nr = M.ncols(), M.nrows()
    A = copy(M.parent().zero())
    for r in range(nr):
        for c in range(nc):
            A[r, c] = M[r, c]**exponent
    return A


def matrix_power_prime(M, exponent=2):
    """Derivative of matrix_power."""
    nc, nr = M.ncols(), M.nrows()
    A = copy(M.parent().zero())
    for r in range(nr):
        for c in range(nc):
            A[r, c] = exponent * M[r, c]**(exponent - 1)
    return A


def elementwise_product(M, N):
    """Element-wise product of M and N."""
    nc, nr = M.ncols(), M.nrows()
    A = copy(M.parent().zero())
    for r in range(nr):
        for c in range(nc):
            A[r, c] = M[r, c] * N[r, c]
    return A


def monomial_list(v, k):
    """Return a list of all monomials in the entries of v of degree k."""
    nvars = len(v)
    exponents_list = list(WeightedIntegerVectors(k, [1 for t in v]))
    return [prod([v[i] ** exponents[i] for i in range(nvars)]) for exponents in exponents_list]


def compute_dimension(network_widths, network_exponent):
    """Compute the dimension of the neurovariety of a network with arbitrary output dimension.
    We use more than one large prime for safety.
    """

    primes = [8191] 
             
    dims = []
    for p in primes:
        nn = Network(network_widths, network_exponent, p)
        num_params = sum([m * n for m, n in zip(nn.sizes[:-1], nn.sizes[1:])])
        degree = nn.degree
        dim_poly_vector = binomial(nn.degree + nn.sizes[0] - 1, nn.sizes[0] - 1)
        nsamples = 2 * dim_poly_vector  # nsamples should be at least dim_poly_vector
        x = random_matrix(nn.FF, nn.sizes[0], nsamples)
        monomials = matrix(nn.FF, [monomial_list(v, degree) for v in x.T])
        monomials_pinv = monomials.pseudoinverse()
        jacobian_matrix = zero_matrix(nn.FF, nn.sizes[-1] * dim_poly_vector, num_params)
        for j in range(nn.sizes[-1]):
            gradients_samples = zero_matrix(nn.FF, nsamples, num_params)
            basis_vec = zero_matrix(nn.FF, nn.sizes[-1], 1)
            basis_vec[j, 0] = 1
            for i in range(nsamples):
                gradient_matrices = nn.backprop(x[:, i], basis_vec)[1]  # use no biases
                gradients_samples[i, :] = matrix(nn.FF, [[t for mat in gradient_matrices for t in mat.list()]])  
            jacobian_matrix[j * dim_poly_vector:(j + 1) * dim_poly_vector, :] = monomials_pinv * gradients_samples
        dims.append(rank(jacobian_matrix))
    if not all(d == dims[0] for d in dims):
        raise ValueError('different dimensions over finite fields: ' + str(dims))
    ambient_dim = (binomial(nn.degree + nn.sizes[0] - 1, nn.sizes[0] - 1)) * nn.sizes[-1]
    naive_bound = sum([(m - 1) * n for m, n in zip(network_widths[:-1], network_widths[1:])]) + network_widths[-1]
    ex_dim = min(ambient_dim, naive_bound)
    return( nn.sizes, # d
            nn.exponent,                         # r
            ambient_dim,                         # ambient_dim
            ex_dim,                              # expected_dim
            int(numpy.mean(dims)),                           # dimension 
            ex_dim - dims[0]                   # defect
           )




if __name__ == "__main__":
    target = [2, 3, 4, 5, 4, 6, 4, 1]
    exponent = 2
    
    # The exact 6 immediate sub-architectures
    sub_architectures = [
        [2, 2, 4, 5, 4, 6, 4, 1],
        [2, 3, 3, 5, 4, 6, 4, 1],
        [2, 3, 4, 4, 4, 6, 4, 1],
        [2, 3, 4, 5, 3, 6, 4, 1],
        [2, 3, 4, 5, 4, 5, 4, 1],
        [2, 3, 4, 5, 4, 6, 3, 1]
    ]

    target_dim = compute_dimension(target, exponent)[4]
    ambient_dim = compute_dimension(target, exponent)[2]
    print(f"Ambient Dimension is {ambient_dim}")
    print(f"  Target {target} -> Dimension: {target_dim}")
    
    is_minimal = True
    for sub in sub_architectures:
        sub_dim = compute_dimension(sub, exponent)[4]
        print(f"  Check {sub} -> Dimension: {sub_dim}")
        
        if sub_dim >= target_dim:
            is_minimal = False

    print("\nRESULT:", "MINIMAL FILLING" if is_minimal else "NOT MINIMAL")

ZeroDivisionError: input matrix must be nonsingular

# Extract Weights 

We run the same computation with $p=101$ and print the weights $\theta$ of the network with architecture $(2,3,4,5,4,6,4,1)$ and $r=2$. The code determines that the Jacobian at $\theta$ as mod $p$ is indeed $65$ and thus exhibits a proof that the architecture is filling.

In [ ]:
from sage.all import *
import itertools
import numpy

class Network(object):

    def __init__(self, sizes, exponent=2, finite_field=100003):
        """The list ``sizes`` contains the number of neurons in the
        respective layers of the network.  For example, if the list
        was [2, 3, 1] then it would be a three-layer network, with the
        first layer containing 2 neurons, the second layer 3 neurons,
        and the third layer 1 neuron.  Currently, biases are initialized to 
        zero and weights are initialized randomly."""
        self.num_layers = len(sizes)
        self.sizes = sizes
        self.FF = GF(finite_field)
        self.biases = [zero_matrix(self.FF, y, 1) for y in sizes[1:]]  # [random_matrix(FF, y, 1) for y in sizes[1:]]
        self.weights = [random_matrix(self.FF, y, x) for x, y in zip(sizes[:-1], sizes[1:])]
        self.exponent = exponent
        self.degree = self.exponent**(self.num_layers - 2)

    def feedforward(self, a):
        """Return the output of the network if ``a`` is input."""
        for b, w in zip(self.biases[:-1], self.weights[:-1]):
            a = matrix_power(w * a + b, self.exponent)
        return self.weights[-1] * a + self.biases[-1]

    def backprop(self, x, pullback_vector=None):
        """Return a tuple ``(nabla_b, nabla_w)`` representing the
        gradient for the *output*pullback_vector* function.  ``nabla_b`` and
        ``nabla_w`` are layer-by-layer lists of matrices, similar
        to ``self.biases`` and ``self.weights``."""

        if pullback_vector is None:
            pullback_vector = ones_matrix(self.FF, self.sizes[-1], 1)

        nabla_b = [zero_matrix(self.FF, b.nrows(), b.ncols()) for b in self.biases]
        nabla_w = [zero_matrix(self.FF, w.nrows(), w.ncols()) for w in self.weights]
        # feedforward
        activation = x
        activations = [x]  # list to store all the activations, layer by layer
        zs = []  # list to store all the z vectors, layer by layer
        for b, w in zip(self.biases, self.weights):
            z = w * activation + b
            zs.append(z)
            activation = matrix_power(z, self.exponent)
            activations.append(activation)
        # backward pass
        delta = pullback_vector
        nabla_b[-1] = delta
        nabla_w[-1] = delta * activations[-2].T
        for l in range(2, self.num_layers):
            z = zs[-l]
            sp = matrix_power_prime(z, self.exponent)
            delta = elementwise_product(self.weights[-l + 1].T * delta, sp)
            nabla_b[-l] = delta
            nabla_w[-l] = delta * activations[-l - 1].T
        return (nabla_b, nabla_w)


def matrix_power(M, exponent=2):
    """Raise elements in M to the power exponent."""
    nc, nr = M.ncols(), M.nrows()
    A = copy(M.parent().zero())
    for r in range(nr):
        for c in range(nc):
            A[r, c] = M[r, c]**exponent
    return A


def matrix_power_prime(M, exponent=2):
    """Derivative of matrix_power."""
    nc, nr = M.ncols(), M.nrows()
    A = copy(M.parent().zero())
    for r in range(nr):
        for c in range(nc):
            A[r, c] = exponent * M[r, c]**(exponent - 1)
    return A


def elementwise_product(M, N):
    """Element-wise product of M and N."""
    nc, nr = M.ncols(), M.nrows()
    A = copy(M.parent().zero())
    for r in range(nr):
        for c in range(nc):
            A[r, c] = M[r, c] * N[r, c]
    return A


def monomial_list(v, k):
    """Return a list of all monomials in the entries of v of degree k."""
    nvars = len(v)
    exponents_list = list(WeightedIntegerVectors(k, [1 for t in v]))
    return [prod([v[i] ** exponents[i] for i in range(nvars)]) for exponents in exponents_list]


def compute_dimension(network_widths, network_exponent):
    """Compute the dimension of the neurovariety of a network with arbitrary output dimension.
    We use more than one large prime for safety.
    """

    primes = [101] 
             
    dims = []
    for p in primes:
        nn = Network(network_widths, network_exponent, p)
        num_params = sum([m * n for m, n in zip(nn.sizes[:-1], nn.sizes[1:])])
        degree = nn.degree
        dim_poly_vector = binomial(nn.degree + nn.sizes[0] - 1, nn.sizes[0] - 1)
        nsamples = 2 * dim_poly_vector  # nsamples should be at least dim_poly_vector
        x = random_matrix(nn.FF, nn.sizes[0], nsamples)
        monomials = matrix(nn.FF, [monomial_list(v, degree) for v in x.T])
        monomials_pinv = monomials.pseudoinverse()
        jacobian_matrix = zero_matrix(nn.FF, nn.sizes[-1] * dim_poly_vector, num_params)
        for j in range(nn.sizes[-1]):
            gradients_samples = zero_matrix(nn.FF, nsamples, num_params)
            basis_vec = zero_matrix(nn.FF, nn.sizes[-1], 1)
            basis_vec[j, 0] = 1
            for i in range(nsamples):
                gradient_matrices = nn.backprop(x[:, i], basis_vec)[1]  # use no biases
                gradients_samples[i, :] = matrix(nn.FF, [[t for mat in gradient_matrices for t in mat.list()]])  
            jacobian_matrix[j * dim_poly_vector:(j + 1) * dim_poly_vector, :] = monomials_pinv * gradients_samples
        dims.append(rank(jacobian_matrix))
    if not all(d == dims[0] for d in dims):
        raise ValueError('different dimensions over finite fields: ' + str(dims))
    ambient_dim = (binomial(nn.degree + nn.sizes[0] - 1, nn.sizes[0] - 1)) * nn.sizes[-1]
    naive_bound = sum([(m - 1) * n for m, n in zip(network_widths[:-1], network_widths[1:])]) + network_widths[-1]
    ex_dim = min(ambient_dim, naive_bound)
    return( nn.sizes, # d
            nn.exponent,                         # r
            ambient_dim,                         # ambient_dim
            ex_dim,                              # expected_dim
            int(numpy.mean(dims)),                           # dimension 
            ex_dim - dims[0]                   # defect
           )


if __name__ == "__main__":
    set_random_seed(42)
    
    target = [2, 3, 4, 5, 4, 6, 4, 1]
    exponent = 2
    p = 101
    
    # Create network for target to capture weights
    target_network = Network(target, exponent, finite_field=p)
    
    # Mod p all weights and biases before network runs
    for i in range(len(target_network.weights)):
        for r in range(target_network.weights[i].nrows()):
            for c in range(target_network.weights[i].ncols()):
                target_network.weights[i][r, c] = target_network.weights[i][r, c] % p
    
    for i in range(len(target_network.biases)):
        for r in range(target_network.biases[i].nrows()):
            for c in range(target_network.biases[i].ncols()):
                target_network.biases[i][r, c] = target_network.biases[i][r, c] % p
    
    print("=" * 80)
    print(f"TARGET NETWORK: {target}")
    print(f"Exponent: {exponent}")
    print(f"Finite Field: GF({p})")
    print(f"Seed: 42")
    print(f"Weights reduced mod {p}")
    print("=" * 80)
    print("\nWEIGHTS (mod p):")
    for layer_idx, weight_matrix in enumerate(target_network.weights):
        input_dim = target[layer_idx]
        output_dim = target[layer_idx + 1]
        print(f"\nLayer {layer_idx} -> {layer_idx + 1} (shape: {output_dim} x {input_dim}):")
        print(weight_matrix)
    print("\n" + "=" * 80)
    
    # The exact 6 immediate sub-architectures
    sub_architectures = [
        [2, 2, 4, 5, 4, 6, 4, 1],
        [2, 3, 3, 5, 4, 6, 4, 1],
        [2, 3, 4, 4, 4, 6, 4, 1],
        [2, 3, 4, 5, 3, 6, 4, 1],
        [2, 3, 4, 5, 4, 5, 4, 1],
        [2, 3, 4, 5, 4, 6, 3, 1]
    ]

    # Reset seed for compute_dimension calls
    set_random_seed(42)
    target_dim = compute_dimension(target, exponent)[4]
    ambient_dim = compute_dimension(target, exponent)[2]
    print(f"Ambient Dimension is {ambient_dim}")
    print(f"  Target {target} -> Dimension: {target_dim}")
    
    is_minimal = True
    for sub in sub_architectures:
        sub_dim = compute_dimension(sub, exponent)[4]
        print(f"  Check {sub} -> Dimension: {sub_dim}")
        
        if sub_dim >= target_dim:
            is_minimal = False

    print("\nRESULT:", "MINIMAL FILLING" if is_minimal else "NOT MINIMAL")

TARGET NETWORK: [2, 3, 4, 5, 4, 6, 4, 1]
Exponent: 2
Finite Field: GF(101)
Seed: 42
Weights reduced mod 101

WEIGHTS (mod p):

Layer 0 -> 1 (shape: 3 x 2):
[94  6]
[ 9  7]
[58 11]

Layer 1 -> 2 (shape: 4 x 3):
[96 20 67]
[83 52 74]
[30 41  2]
[ 3 90  1]

Layer 2 -> 3 (shape: 5 x 4):
[23 36 99 71]
[27 94 56  3]
[91 50 65 62]
[24 20 30 76]
[41 49 76 81]

Layer 3 -> 4 (shape: 4 x 5):
[ 94  55  89  90  60]
[100  41   6  49  83]
[ 78  56  50   3  56]
[ 16  46  15  10  16]

Layer 4 -> 5 (shape: 6 x 4):
[51 37 71 85]
[97 23 57 56]
[12 47 41 60]
[31 60 63 15]
[54  2 57 60]
[40  2 34 83]

Layer 5 -> 6 (shape: 4 x 6):
[57 90  8 19 55 36]
[ 8 35 94 89 25 36]
[ 8 55  6 69 14 28]
[59  7 85 82 60  4]

Layer 6 -> 7 (shape: 1 x 4):
[99 42 87 98]

Ambient Dimension is 65
  Target [2, 3, 4, 5, 4, 6, 4, 1] -> Dimension: 65
  Check [2, 2, 4, 5, 4, 6, 4, 1] -> Dimension: 35
  Check [2, 3, 3, 5, 4, 6, 4, 1] -> Dimension: 60
  Check [2, 3, 4, 4, 4, 6, 4, 1] -> Dimension: 62
  Check [2, 3, 4, 5, 3, 6, 4, 1] -